# 딥페이크 범죄 대응을 위한 AI 탐지 모델 경진대회

## 📋 대회 정보
- **목표**: 이미지 및 동영상 속 얼굴의 딥페이크 여부를 판별하는 이진 분류 모델 개발
- **평가지표**: **Macro F1-score**
- **입력**: 이미지 (동영상은 프레임 추출)
- **출력**: Fake(1) or Real(0)
- **데이터**: 얼굴 1명 포함, 동영상 평균 5초, 다양한 인종/연령

## 🎯 학습 전략
1. **데이터셋**: FaceForensics++, Celeb-DF, DFDC, WildDeepfake 병합
2. **모델**: EfficientNet-B4, XceptionNet 앙상블 + 주파수 도메인 분석
3. **크로스 데이터셋 학습**으로 일반화 성능 향상
4. **5-Fold Cross Validation**으로 안정성 확보

---

## 1. 환경 설정 및 라이브러리 설치

In [ ]:
# 필수 라이브러리 설치
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install timm albumentations opencv-python-headless facenet-pytorch
!pip install scikit-learn pandas numpy matplotlib seaborn tqdm
!pip install pytorchvideo av ffmpeg-python
!pip install efficientnet-pytorch

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models

import albumentations as A
from albumentations.pytorch import ToTensorV2
from facenet_pytorch import MTCNN

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

import timm
from efficientnet_pytorch import EfficientNet

# 시드 고정
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. 데이터셋 다운로드 가이드

### 📥 추천 데이터셋 다운로드

#### 1. **FaceForensics++ (필수)**
- **다운로드**: https://github.com/ondyari/FaceForensics
- 신청 양식 작성 후 다운로드 링크 수신
- 크기: ~500GB (압축 버전 사용 권장)
- **사용 데이터**: Deepfakes, FaceSwap (학습 80%, 테스트 20%)

#### 2. **Celeb-DF v2 (필수)**
- **다운로드**: https://github.com/yuezunli/celeb-deepfakeforensics
- 크기: ~590 원본 + 5,639 딥페이크 비디오
- **사용**: 학습 70%, 테스트 30%

#### 3. **DFDC (Deepfake Detection Challenge) (권장)**
- **Kaggle**: https://www.kaggle.com/c/deepfake-detection-challenge
- Preview 데이터셋 사용 (5,000 비디오)
- **사용**: 학습 80%, 테스트 20%

#### 4. **WildDeepfake (추론용)**
- **GitHub**: https://github.com/OpenTAI/wild-deepfake
- 실제 인터넷 유통 딥페이크 707개
- **사용**: 추론 테스트 (일반화 평가)

### 📁 디렉토리 구조
```
datasets/
├── faceforensics/
│   ├── real/
│   └── fake/
├── celebdf/
│   ├── real/
│   └── fake/
├── dfdc/
│   ├── real/
│   └── fake/
└── wilddeepfake/
    ├── real/
    └── fake/
```

## 3. 전처리 파이프라인

### 3.1 동영상 → 프레임 추출 및 얼굴 검출

In [ ]:
class VideoFrameExtractor:
    """
    동영상에서 프레임을 추출하고 얼굴을 검출하는 클래스
    
    특징:
    - MTCNN으로 얼굴 검출 (정확도 높음)
    - 5fps 샘플링 (5초 영상 → 약 25프레임)
    - 얼굴 크롭 및 정렬
    """
    def __init__(self, output_size=224, fps=5):
        self.output_size = output_size
        self.fps = fps
        self.mtcnn = MTCNN(
            image_size=output_size,
            margin=20,
            device=device,
            post_process=False  # 원본 유지
        )
    
    def extract_frames(self, video_path, max_frames=30):
        """
        동영상에서 프레임 추출
        
        Args:
            video_path: 동영상 파일 경로
            max_frames: 최대 추출 프레임 수
        
        Returns:
            frames: 추출된 프레임 리스트 (RGB)
        """
        frames = []
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"Error: Cannot open video {video_path}")
            return frames
        
        # 비디오 정보
        video_fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # 샘플링 간격 계산
        frame_interval = int(video_fps / self.fps)
        frame_count = 0
        
        while len(frames) < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            if frame_count % frame_interval == 0:
                # BGR → RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame_rgb)
            
            frame_count += 1
        
        cap.release()
        return frames
    
    def detect_and_crop_face(self, frame):
        """
        프레임에서 얼굴 검출 및 크롭
        
        Args:
            frame: RGB 이미지 (H, W, 3)
        
        Returns:
            cropped_face: 크롭된 얼굴 이미지 (224, 224, 3)
        """
        try:
            # MTCNN으로 얼굴 검출
            face = self.mtcnn(frame)
            
            if face is not None:
                # Tensor → NumPy (C, H, W) → (H, W, C)
                face_np = face.permute(1, 2, 0).cpu().numpy()
                # 정규화 해제 [0, 1] → [0, 255]
                face_np = (face_np * 255).astype(np.uint8)
                return face_np
            else:
                # 얼굴 검출 실패 시 중앙 크롭
                h, w = frame.shape[:2]
                size = min(h, w)
                y1 = (h - size) // 2
                x1 = (w - size) // 2
                center_crop = frame[y1:y1+size, x1:x1+size]
                return cv2.resize(center_crop, (self.output_size, self.output_size))
        
        except Exception as e:
            print(f"Error in face detection: {e}")
            # 중앙 크롭으로 폴백
            h, w = frame.shape[:2]
            size = min(h, w)
            y1 = (h - size) // 2
            x1 = (w - size) // 2
            center_crop = frame[y1:y1+size, x1:x1+size]
            return cv2.resize(center_crop, (self.output_size, self.output_size))
    
    def process_video(self, video_path, output_dir, video_id, label):
        """
        동영상 전체 처리 파이프라인
        
        Args:
            video_path: 동영상 파일 경로
            output_dir: 출력 디렉토리
            video_id: 비디오 ID
            label: 레이블 (0=real, 1=fake)
        
        Returns:
            processed_frames: 처리된 프레임 수
        """
        frames = self.extract_frames(video_path)
        processed_frames = 0
        
        label_dir = 'fake' if label == 1 else 'real'
        save_dir = os.path.join(output_dir, label_dir)
        os.makedirs(save_dir, exist_ok=True)
        
        for idx, frame in enumerate(frames):
            face = self.detect_and_crop_face(frame)
            if face is not None:
                # 저장
                output_path = os.path.join(save_dir, f"{video_id}_frame{idx:03d}.jpg")
                cv2.imwrite(output_path, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
                processed_frames += 1
        
        return processed_frames

# 사용 예시
extractor = VideoFrameExtractor(output_size=224, fps=5)
print("VideoFrameExtractor 준비 완료!")

In [ ]:
# 데이터셋 전처리 (실행 예시)
def preprocess_dataset(dataset_path, output_path):
    """
    데이터셋 전체 전처리
    
    실행 시간: 약 2-4시간 (GPU 사용 시)
    """
    extractor = VideoFrameExtractor(output_size=224, fps=5)
    
    # Real 비디오 처리
    real_videos = [f for f in os.listdir(os.path.join(dataset_path, 'real')) if f.endswith('.mp4')]
    print(f"Processing {len(real_videos)} real videos...")
    
    for video_file in tqdm(real_videos[:100], desc="Real videos"):  # 샘플 100개
        video_path = os.path.join(dataset_path, 'real', video_file)
        video_id = os.path.splitext(video_file)[0]
        extractor.process_video(video_path, output_path, video_id, label=0)
    
    # Fake 비디오 처리
    fake_videos = [f for f in os.listdir(os.path.join(dataset_path, 'fake')) if f.endswith('.mp4')]
    print(f"Processing {len(fake_videos)} fake videos...")
    
    for video_file in tqdm(fake_videos[:100], desc="Fake videos"):  # 샘플 100개
        video_path = os.path.join(dataset_path, 'fake', video_file)
        video_id = os.path.splitext(video_file)[0]
        extractor.process_video(video_path, output_path, video_id, label=1)
    
    print(f"전처리 완료! 저장 위치: {output_path}")

# 실행 (주석 해제 후 사용)
# preprocess_dataset('datasets/faceforensics', 'processed_data/faceforensics')
# preprocess_dataset('datasets/celebdf', 'processed_data/celebdf')

### 3.2 데이터 증강 (Albumentations)

In [ ]:
def get_transforms(phase='train'):
    """
    데이터 증강 파이프라인
    
    주요 증강:
    - 주파수 도메인 노이즈 추가 (압축 아티팩트 시뮬레이션)
    - 색상/밝기 변형 (다양한 조명 조건)
    - 기하학적 변형 (회전, 플립)
    """
    if phase == 'train':
        return A.Compose([
            # 기하학적 변형
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1,
                scale_limit=0.1,
                rotate_limit=15,
                p=0.5
            ),
            
            # 색상 및 밝기
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1),
            ], p=0.5),
            
            # 압축 및 품질 저하 (실제 환경 시뮬레이션)
            A.OneOf([
                A.ImageCompression(quality_lower=60, quality_upper=100, p=1),
                A.GaussNoise(var_limit=(10, 50), p=1),
                A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1),
            ], p=0.3),
            
            # 블러 (다양한 카메라 품질)
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 5), p=1),
                A.MotionBlur(blur_limit=(3, 5), p=1),
            ], p=0.2),
            
            # 정규화
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
            ToTensorV2()
        ])
    
    else:  # validation/test
        return A.Compose([
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
            ToTensorV2()
        ])

# 테스트
train_transform = get_transforms('train')
val_transform = get_transforms('val')
print("데이터 증강 파이프라인 준비 완료!")

### 3.3 커스텀 데이터셋

In [ ]:
class DeepfakeDataset(Dataset):
    """
    딥페이크 감지를 위한 커스텀 데이터셋
    
    구조:
    - 이미지 기반 (동영상은 프레임 추출 후 사용)
    - Real(0) vs Fake(1) 이진 분류
    - 클래스 균형 유지 (SMOTE 또는 가중치 샘플링)
    """
    def __init__(self, data_dir, transform=None, phase='train'):
        self.data_dir = data_dir
        self.transform = transform
        self.phase = phase
        
        # 이미지 경로 및 레이블 수집
        self.images = []
        self.labels = []
        
        # Real 이미지
        real_dir = os.path.join(data_dir, 'real')
        if os.path.exists(real_dir):
            real_images = [os.path.join(real_dir, f) for f in os.listdir(real_dir) 
                          if f.endswith(('.jpg', '.png'))]
            self.images.extend(real_images)
            self.labels.extend([0] * len(real_images))
        
        # Fake 이미지
        fake_dir = os.path.join(data_dir, 'fake')
        if os.path.exists(fake_dir):
            fake_images = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) 
                          if f.endswith(('.jpg', '.png'))]
            self.images.extend(fake_images)
            self.labels.extend([1] * len(fake_images))
        
        print(f"{phase} dataset: {len(self.images)} images")
        print(f"Real: {self.labels.count(0)}, Fake: {self.labels.count(1)}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # 이미지 로드
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # 증강 적용
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, label

# 동영상용 데이터셋 (프레임 집계)
class VideoDataset(Dataset):
    """
    동영상 레벨 예측을 위한 데이터셋
    
    특징:
    - 비디오당 여러 프레임을 로드
    - 프레임별 예측 → 투표(majority vote) 또는 평균
    """
    def __init__(self, video_dir, frame_dir, transform=None, frames_per_video=10):
        self.video_dir = video_dir
        self.frame_dir = frame_dir
        self.transform = transform
        self.frames_per_video = frames_per_video
        
        # 비디오별 프레임 그룹화
        self.video_groups = {}
        for label_dir in ['real', 'fake']:
            dir_path = os.path.join(frame_dir, label_dir)
            if os.path.exists(dir_path):
                for img_file in os.listdir(dir_path):
                    video_id = '_'.join(img_file.split('_')[:-1])  # video_id_frame001.jpg
                    if video_id not in self.video_groups:
                        self.video_groups[video_id] = {'frames': [], 'label': 0 if label_dir == 'real' else 1}
                    self.video_groups[video_id]['frames'].append(os.path.join(dir_path, img_file))
        
        self.video_ids = list(self.video_groups.keys())
        print(f"Total videos: {len(self.video_ids)}")
    
    def __len__(self):
        return len(self.video_ids)
    
    def __getitem__(self, idx):
        video_id = self.video_ids[idx]
        video_data = self.video_groups[video_id]
        
        # 랜덤으로 프레임 선택
        frame_paths = np.random.choice(
            video_data['frames'], 
            size=min(self.frames_per_video, len(video_data['frames'])),
            replace=False
        )
        
        frames = []
        for frame_path in frame_paths:
            image = cv2.imread(frame_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            
            frames.append(image)
        
        # 프레임 스택
        frames_tensor = torch.stack(frames)  # (num_frames, C, H, W)
        label = video_data['label']
        
        return frames_tensor, label, video_id

print("데이터셋 클래스 준비 완료!")

## 4. 모델 아키텍처

### 4.1 EfficientNet-B4 백본

In [ ]:
class EfficientNetDeepfakeDetector(nn.Module):
    """
    EfficientNet-B4 기반 딥페이크 감지 모델
    
    특징:
    - ImageNet 사전학습 가중치 사용
    - Dropout으로 과적합 방지
    - 이진 분류 (Sigmoid)
    """
    def __init__(self, pretrained=True, dropout=0.5):
        super().__init__()
        
        # EfficientNet-B4 백본
        if pretrained:
            self.backbone = EfficientNet.from_pretrained('efficientnet-b4')
        else:
            self.backbone = EfficientNet.from_name('efficientnet-b4')
        
        # 분류 헤드 교체
        num_features = self.backbone._fc.in_features
        self.backbone._fc = nn.Identity()  # 원래 FC 제거
        
        # 새로운 분류 헤드
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        features = self.backbone(x)
        output = self.classifier(features)
        return output.squeeze(1)  # (batch_size, 1) → (batch_size,)

# 모델 초기화
model_efficientnet = EfficientNetDeepfakeDetector(pretrained=True, dropout=0.5)
model_efficientnet = model_efficientnet.to(device)
print(f"EfficientNet-B4 모델 파라미터: {sum(p.numel() for p in model_efficientnet.parameters()):,}")

### 4.2 XceptionNet 백본

In [ ]:
class XceptionDeepfakeDetector(nn.Module):
    """
    Xception 기반 딥페이크 감지 모델
    
    장점:
    - FaceForensics++ 벤치마크에서 우수한 성능
    - 주파수 도메인 아티팩트 검출에 효과적
    """
    def __init__(self, pretrained=True, dropout=0.5):
        super().__init__()
        
        # Xception 백본 (timm 라이브러리)
        self.backbone = timm.create_model(
            'xception',
            pretrained=pretrained,
            num_classes=0  # 분류 헤드 제거
        )
        
        num_features = self.backbone.num_features
        
        # 분류 헤드
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        features = self.backbone(x)
        output = self.classifier(features)
        return output.squeeze(1)

# 모델 초기화
model_xception = XceptionDeepfakeDetector(pretrained=True, dropout=0.5)
model_xception = model_xception.to(device)
print(f"Xception 모델 파라미터: {sum(p.numel() for p in model_xception.parameters()):,}")

### 4.3 주파수 도메인 증강 모델 (FFT 기반)

In [ ]:
class FrequencyAwareDeepfakeDetector(nn.Module):
    """
    주파수 도메인 분석을 포함한 딥페이크 감지 모델
    
    특징:
    - RGB 공간 + 주파수 도메인 듀얼 경로
    - FFT로 고주파 아티팩트 검출
    - 2025년 최신 연구 기법
    """
    def __init__(self, backbone='efficientnet', pretrained=True):
        super().__init__()
        
        # RGB 경로
        if backbone == 'efficientnet':
            if pretrained:
                self.rgb_backbone = EfficientNet.from_pretrained('efficientnet-b4')
            else:
                self.rgb_backbone = EfficientNet.from_name('efficientnet-b4')
            rgb_features = self.rgb_backbone._fc.in_features
            self.rgb_backbone._fc = nn.Identity()
        else:
            self.rgb_backbone = timm.create_model('xception', pretrained=pretrained, num_classes=0)
            rgb_features = self.rgb_backbone.num_features
        
        # 주파수 경로 (간단한 CNN)
        self.freq_conv = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1)
        )
        freq_features = 128
        
        # 융합 분류 헤드
        self.fusion = nn.Sequential(
            nn.Linear(rgb_features + freq_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )
    
    def extract_frequency(self, x):
        """
        FFT로 주파수 도메인 특징 추출
        """
        # RGB → 그레이스케일
        gray = 0.299 * x[:, 0:1, :, :] + 0.587 * x[:, 1:2, :, :] + 0.114 * x[:, 2:3, :, :]
        
        # FFT
        fft = torch.fft.fft2(gray)
        fft_shift = torch.fft.fftshift(fft)
        
        # 진폭 스펙트럼
        magnitude = torch.abs(fft_shift)
        log_magnitude = torch.log(magnitude + 1)  # 로그 스케일
        
        # 3채널로 복제 (Conv2d 입력용)
        freq_image = log_magnitude.repeat(1, 3, 1, 1)
        
        return freq_image
    
    def forward(self, x):
        # RGB 경로
        rgb_features = self.rgb_backbone(x)
        
        # 주파수 경로
        freq_input = self.extract_frequency(x)
        freq_features = self.freq_conv(freq_input)
        freq_features = freq_features.view(freq_features.size(0), -1)
        
        # 융합
        combined = torch.cat([rgb_features, freq_features], dim=1)
        output = self.fusion(combined)
        
        return output.squeeze(1)

# 모델 초기화
model_freq = FrequencyAwareDeepfakeDetector(backbone='efficientnet', pretrained=True)
model_freq = model_freq.to(device)
print(f"Frequency-Aware 모델 파라미터: {sum(p.numel() for p in model_freq.parameters()):,}")

## 5. 학습 파이프라인

### 5.1 손실 함수 및 메트릭

In [ ]:
def calculate_metrics(predictions, labels):
    """
    평가 메트릭 계산
    
    대회 주요 지표: Macro F1-score
    """
    # 이진화 (threshold=0.5)
    preds_binary = (predictions >= 0.5).astype(int)
    
    # Macro F1-score (Real과 Fake의 F1을 평균)
    macro_f1 = f1_score(labels, preds_binary, average='macro')
    
    # 클래스별 F1-score
    f1_per_class = f1_score(labels, preds_binary, average=None)
    
    # Accuracy
    accuracy = accuracy_score(labels, preds_binary)
    
    return {
        'macro_f1': macro_f1,
        'f1_real': f1_per_class[0],
        'f1_fake': f1_per_class[1],
        'accuracy': accuracy
    }

# Focal Loss (클래스 불균형 대응)
class FocalLoss(nn.Module):
    """
    Focal Loss for imbalanced classification
    
    수식: FL(pt) = -α(1-pt)^γ * log(pt)
    """
    def __init__(self, alpha=0.25, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

# 가중치 BCE Loss (클래스 불균형 대응)
def get_weighted_bce_loss(pos_weight):
    """
    클래스 불균형 고려한 BCE Loss
    
    pos_weight: Fake 클래스 가중치 (예: Real이 많으면 pos_weight > 1)
    """
    return nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))

print("손실 함수 및 메트릭 준비 완료!")

### 5.2 학습 함수

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch):
    """
    1 에포크 학습
    """
    model.train()
    running_loss = 0.0
    all_predictions = []
    all_labels = []
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]")
    
    for images, labels in progress_bar:
        images = images.to(device)
        labels = labels.float().to(device)
        
        # Forward
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        # 메트릭 저장
        running_loss += loss.item() * images.size(0)
        all_predictions.extend(outputs.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # 진행률 업데이트
        progress_bar.set_postfix({'loss': loss.item()})
    
    epoch_loss = running_loss / len(dataloader.dataset)
    metrics = calculate_metrics(np.array(all_predictions), np.array(all_labels))
    
    return epoch_loss, metrics

def validate(model, dataloader, criterion, device, epoch):
    """
    검증
    """
    model.eval()
    running_loss = 0.0
    all_predictions = []
    all_labels = []
    
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]")
    
    with torch.no_grad():
        for images, labels in progress_bar:
            images = images.to(device)
            labels = labels.float().to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            all_predictions.extend(outputs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            progress_bar.set_postfix({'loss': loss.item()})
    
    epoch_loss = running_loss / len(dataloader.dataset)
    metrics = calculate_metrics(np.array(all_predictions), np.array(all_labels))
    
    return epoch_loss, metrics

print("학습 함수 준비 완료!")

### 5.3 5-Fold Cross Validation 학습

In [ ]:
def train_kfold(data_dir, model_class, model_params, n_splits=5, epochs=30, batch_size=32, lr=1e-4):
    """
    5-Fold Cross Validation 학습
    
    Args:
        data_dir: 전처리된 데이터 디렉토리
        model_class: 모델 클래스 (EfficientNetDeepfakeDetector 등)
        model_params: 모델 초기화 파라미터
        n_splits: Fold 개수
        epochs: 에포크 수
        batch_size: 배치 크기
        lr: 학습률
    
    Returns:
        fold_results: Fold별 결과
    """
    # 전체 데이터 로드
    full_dataset = DeepfakeDataset(data_dir, transform=None, phase='train')
    
    # 5-Fold 분할
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(full_dataset.images, full_dataset.labels)):
        print(f"\n{'='*50}")
        print(f"Fold {fold + 1}/{n_splits}")
        print(f"{'='*50}")
        
        # Train/Val Split
        train_images = [full_dataset.images[i] for i in train_idx]
        train_labels = [full_dataset.labels[i] for i in train_idx]
        val_images = [full_dataset.images[i] for i in val_idx]
        val_labels = [full_dataset.labels[i] for i in val_idx]
        
        # 데이터로더
        train_dataset = DeepfakeDataset(data_dir, transform=get_transforms('train'))
        train_dataset.images = train_images
        train_dataset.labels = train_labels
        
        val_dataset = DeepfakeDataset(data_dir, transform=get_transforms('val'))
        val_dataset.images = val_images
        val_dataset.labels = val_labels
        
        train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True
        )
        val_loader = DataLoader(
            val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True
        )
        
        # 모델 초기화
        model = model_class(**model_params).to(device)
        
        # 손실 함수 (Focal Loss 사용)
        criterion = FocalLoss(alpha=0.25, gamma=2)
        
        # 옵티마이저 (AdamW + Cosine Annealing)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
        
        # 학습
        best_macro_f1 = 0.0
        best_model_path = f'best_model_fold{fold+1}.pth'
        
        fold_history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}
        
        for epoch in range(1, epochs + 1):
            train_loss, train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
            val_loss, val_metrics = validate(model, val_loader, criterion, device, epoch)
            scheduler.step()
            
            fold_history['train_loss'].append(train_loss)
            fold_history['val_loss'].append(val_loss)
            fold_history['val_macro_f1'].append(val_metrics['macro_f1'])
            
            print(f"\nEpoch {epoch}/{epochs}")
            print(f"Train Loss: {train_loss:.4f}, Train Macro F1: {train_metrics['macro_f1']:.4f}")
            print(f"Val Loss: {val_loss:.4f}, Val Macro F1: {val_metrics['macro_f1']:.4f}")
            print(f"Val F1 Real: {val_metrics['f1_real']:.4f}, Val F1 Fake: {val_metrics['f1_fake']:.4f}")
            
            # Best 모델 저장
            if val_metrics['macro_f1'] > best_macro_f1:
                best_macro_f1 = val_metrics['macro_f1']
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'macro_f1': best_macro_f1,
                }, best_model_path)
                print(f"✓ Best model saved (Macro F1: {best_macro_f1:.4f})")
        
        fold_results.append({
            'fold': fold + 1,
            'best_macro_f1': best_macro_f1,
            'history': fold_history,
            'model_path': best_model_path
        })
    
    # 최종 결과 요약
    print(f"\n{'='*50}")
    print("5-Fold Cross Validation 결과")
    print(f"{'='*50}")
    
    macro_f1_scores = [result['best_macro_f1'] for result in fold_results]
    print(f"Fold별 Best Macro F1: {macro_f1_scores}")
    print(f"평균 Macro F1: {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}")
    
    return fold_results

print("5-Fold CV 학습 함수 준비 완료!")

### 5.4 학습 실행 예시

In [ ]:
# 학습 실행 (주석 해제 후 사용)
"""
# 1. EfficientNet-B4 학습
efficientnet_results = train_kfold(
    data_dir='processed_data/combined',  # FaceForensics++ + Celeb-DF 병합
    model_class=EfficientNetDeepfakeDetector,
    model_params={'pretrained': True, 'dropout': 0.5},
    n_splits=5,
    epochs=30,
    batch_size=32,
    lr=1e-4
)

# 2. Xception 학습
xception_results = train_kfold(
    data_dir='processed_data/combined',
    model_class=XceptionDeepfakeDetector,
    model_params={'pretrained': True, 'dropout': 0.5},
    n_splits=5,
    epochs=30,
    batch_size=32,
    lr=1e-4
)

# 3. Frequency-Aware 학습
freq_results = train_kfold(
    data_dir='processed_data/combined',
    model_class=FrequencyAwareDeepfakeDetector,
    model_params={'backbone': 'efficientnet', 'pretrained': True},
    n_splits=5,
    epochs=30,
    batch_size=16,  # 메모리 절약
    lr=1e-4
)
"""

print("학습 실행 코드 준비 완료!")
print("실제 학습 시 위 코드의 주석을 해제하고 실행하세요.")

## 6. 추론 (Inference)

### 6.1 동영상 레벨 추론

In [ ]:
class DeepfakeInference:
    """
    딥페이크 감지 추론 클래스
    
    특징:
    - 동영상 입력 → 프레임별 예측 → 집계 (Majority Vote or Average)
    - 모델 앙상블 지원
    - 배치 처리 최적화
    """
    def __init__(self, models, device='cuda'):
        """
        Args:
            models: 모델 리스트 (앙상블용)
            device: 디바이스
        """
        self.models = models
        self.device = device
        self.extractor = VideoFrameExtractor(output_size=224, fps=5)
        self.transform = get_transforms('val')
        
        # 모델을 평가 모드로
        for model in self.models:
            model.eval()
    
    def predict_video(self, video_path, aggregation='average'):
        """
        단일 동영상 예측
        
        Args:
            video_path: 동영상 경로
            aggregation: 집계 방법 ('average' or 'majority')
        
        Returns:
            prediction: 0 (Real) or 1 (Fake)
            confidence: 신뢰도 (0~1)
        """
        # 프레임 추출
        frames = self.extractor.extract_frames(video_path, max_frames=30)
        
        if len(frames) == 0:
            print(f"Warning: No frames extracted from {video_path}")
            return 1, 0.5  # 기본값
        
        # 얼굴 검출 및 전처리
        frame_predictions = []
        
        for frame in frames:
            face = self.extractor.detect_and_crop_face(frame)
            
            # 전처리
            augmented = self.transform(image=face)
            image_tensor = augmented['image'].unsqueeze(0).to(self.device)
            
            # 모델 앙상블 예측
            model_outputs = []
            with torch.no_grad():
                for model in self.models:
                    output = model(image_tensor)
                    model_outputs.append(output.item())
            
            # 앙상블 평균
            frame_pred = np.mean(model_outputs)
            frame_predictions.append(frame_pred)
        
        # 집계
        if aggregation == 'average':
            # 평균 확률
            avg_prob = np.mean(frame_predictions)
            prediction = 1 if avg_prob >= 0.5 else 0
            confidence = avg_prob if prediction == 1 else (1 - avg_prob)
        
        elif aggregation == 'majority':
            # Majority Vote
            frame_classes = [1 if p >= 0.5 else 0 for p in frame_predictions]
            prediction = 1 if sum(frame_classes) > len(frame_classes) / 2 else 0
            confidence = sum(frame_classes) / len(frame_classes)
        
        return prediction, confidence
    
    def predict_batch(self, video_paths, output_csv='predictions.csv'):
        """
        배치 예측 (대회 제출용)
        
        Args:
            video_paths: 동영상 경로 리스트
            output_csv: 출력 CSV 파일명
        
        Returns:
            df: 예측 결과 DataFrame
        """
        results = []
        
        for video_path in tqdm(video_paths, desc="Inference"):
            video_id = os.path.basename(video_path)
            prediction, confidence = self.predict_video(video_path)
            
            results.append({
                'video_id': video_id,
                'prediction': prediction,
                'confidence': confidence
            })
        
        df = pd.DataFrame(results)
        df.to_csv(output_csv, index=False)
        print(f"✓ 예측 결과 저장: {output_csv}")
        
        return df

print("추론 클래스 준비 완료!")

### 6.2 추론 실행 예시

In [ ]:
# 추론 실행 (주석 해제 후 사용)
"""
# 모델 로드 (5-Fold 중 Best 모델 사용)
model1 = EfficientNetDeepfakeDetector(pretrained=False)
checkpoint1 = torch.load('best_model_fold1.pth')
model1.load_state_dict(checkpoint1['model_state_dict'])
model1 = model1.to(device)

model2 = XceptionDeepfakeDetector(pretrained=False)
checkpoint2 = torch.load('best_model_xception_fold1.pth')
model2.load_state_dict(checkpoint2['model_state_dict'])
model2 = model2.to(device)

# 추론 클래스 초기화 (앙상블)
inference = DeepfakeInference(models=[model1, model2], device=device)

# 테스트 비디오 예측
test_video_paths = [
    'test_videos/video1.mp4',
    'test_videos/video2.mp4',
    # ...
]

# 배치 예측
predictions_df = inference.predict_batch(test_video_paths, output_csv='submission.csv')
print(predictions_df.head())

# 제출 파일 확인
print(f"\nFake 비율: {predictions_df['prediction'].sum() / len(predictions_df):.2%}")
print(f"평균 신뢰도: {predictions_df['confidence'].mean():.4f}")
"""

print("추론 코드 준비 완료!")
print("실제 추론 시 위 코드의 주석을 해제하고 실행하세요.")

## 7. 평가 및 분석

### 7.1 혼동 행렬 및 성능 분석

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    """
    혼동 행렬 시각화
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Real', 'Fake'], 
                yticklabels=['Real', 'Fake'])
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # 세부 분석
    tn, fp, fn, tp = cm.ravel()
    print(f"True Negatives (Real → Real): {tn}")
    print(f"False Positives (Real → Fake): {fp}")
    print(f"False Negatives (Fake → Real): {fn}")
    print(f"True Positives (Fake → Fake): {tp}")
    print(f"\nPrecision (Fake): {tp / (tp + fp):.4f}")
    print(f"Recall (Fake): {tp / (tp + fn):.4f}")

def plot_training_history(history, title='Training History'):
    """
    학습 히스토리 시각화
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Macro F1
    axes[1].plot(history['val_macro_f1'], label='Val Macro F1', marker='o', color='green')
    axes[1].set_title('Macro F1-score')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Macro F1')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

print("평가 함수 준비 완료!")

## 8. 최적화 팁

### 8.1 학습 전략

#### 1. **데이터 증강**
- 주파수 도메인 노이즈 추가 (FFT)
- JPEG 압축 시뮬레이션
- 밝기/대비 조정 (다양한 조명 조건)

#### 2. **크로스 데이터셋 학습**
- FaceForensics++ + Celeb-DF 병합
- DFDC로 추가 학습 (Fine-tuning)
- WildDeepfake로 실전 테스트

#### 3. **모델 앙상블**
- EfficientNet-B4 + Xception
- Soft Voting (확률 평균)
- 5-Fold 모델 평균

#### 4. **하이퍼파라미터 튜닝**
- Learning Rate: 1e-4 ~ 1e-5
- Batch Size: 16 ~ 64
- Dropout: 0.3 ~ 0.7
- Weight Decay: 1e-4 ~ 1e-5

---

### 8.2 성능 향상 체크리스트

- [ ] **전처리**: MTCNN으로 얼굴 검출 (RetinaFace도 고려)
- [ ] **데이터 증강**: 압축, 노이즈, 밝기 조정
- [ ] **모델**: EfficientNet-B4, Xception 병합
- [ ] **손실 함수**: Focal Loss (클래스 불균형 대응)
- [ ] **학습**: 5-Fold CV, Early Stopping
- [ ] **앙상블**: 여러 모델 + Fold 평균
- [ ] **추론**: 프레임별 예측 → Average 집계
- [ ] **검증**: WildDeepfake로 일반화 테스트

---

### 8.3 추가 고려사항

#### 1. **Test Time Augmentation (TTA)**
```python
def tta_predict(model, image, n_tta=5):
    predictions = []
    for _ in range(n_tta):
        augmented = transform(image=image)['image']
        pred = model(augmented.unsqueeze(0))
        predictions.append(pred.item())
    return np.mean(predictions)
```

#### 2. **Pseudo Labeling** (대회 데이터 활용)
- 샘플 데이터로 초기 학습
- 테스트셋 일부에 Pseudo Label 부여
- 재학습 (Semi-supervised)

#### 3. **Adversarial Training**
- FGSM, PGD로 강건성 향상
- 압축 공격 대응

---

## 9. 참고 자료

### 데이터셋
- FaceForensics++: https://github.com/ondyari/FaceForensics
- Celeb-DF: https://github.com/yuezunli/celeb-deepfakeforensics
- DFDC: https://www.kaggle.com/c/deepfake-detection-challenge
- WildDeepfake: https://github.com/OpenTAI/wild-deepfake
- DeepfakeBench: https://github.com/SCLBD/DeepfakeBench

### 논문
- LNCLIP-DF (2025): https://arxiv.org/abs/2508.06248
- Frequency-Aware Detection (2025): Nature Scientific Reports
- XceptionNet: FaceForensics++ Benchmark

### 라이브러리
- PyTorch: https://pytorch.org/
- Albumentations: https://albumentations.ai/
- MTCNN (facenet-pytorch): https://github.com/timesler/facenet-pytorch
- timm: https://github.com/huggingface/pytorch-image-models

---

## 📊 평가지표 (Macro F1-score)

$$
\text{Macro F1} = \frac{1}{2} \left( F1_{\text{Real}} + F1_{\text{Fake}} \right)
$$

$$
F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

**목표**: Macro F1 ≥ 0.95 (리더보드 상위권)

---

## 🎯 성공 전략 요약

1. **다양한 데이터셋 병합**: FaceForensics++ + Celeb-DF + DFDC
2. **강력한 모델 앙상블**: EfficientNet-B4 + Xception + Frequency-Aware
3. **5-Fold Cross Validation**: 안정적인 성능 확보
4. **Test Time Augmentation**: 추론 시 정확도 향상
5. **크로스 데이터셋 검증**: WildDeepfake로 일반화 테스트

**최종 목표**: Macro F1-score 0.95+ 달성! 🏆